# Firewatch SFT Training - Colab

Thin launcher for the real Hub-backed SFT pipeline. Durable artifacts are written to Hugging Face Hub model repos, not to the Colab VM.

## 1. Runtime Setup

Select a GPU runtime, add `HF_TOKEN` in Colab Secrets, then run each cell in order.

In [ ]:
!nvidia-smi
!pip install -q uv

In [ ]:
from google.colab import userdata
import os

token = userdata.get('HF_TOKEN')
if not token:
    raise RuntimeError('Set HF_TOKEN in Colab Secrets before running training.')
os.environ['HF_TOKEN'] = token
os.environ.setdefault('HF_HUB_DISABLE_TELEMETRY', '1')

## 2. Repository Setup

Clone the project if it is not already mounted. Update `REPO_URL` for your working branch or fork.

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/10doshi12/firewatch_agent.git'
REPO_BRANCH = 'main'
repo_dir = Path('/content/firewatch_agent')
if repo_dir.exists():
    # Always pull on a re-run — without this, a stale clone silently runs out-of-date code.
    !cd {repo_dir} && git fetch origin {REPO_BRANCH} && git reset --hard origin/{REPO_BRANCH}
else:
    !git clone --branch {REPO_BRANCH} {REPO_URL} {repo_dir}
%cd /content/firewatch_agent
!uv sync
# uv pip install defaults to system Python on Colab; force the project venv via --python
# so Unsloth (and anything else we add) lands where `uv run` will actually see it.
!uv pip install --python .venv/bin/python -e '.[fast]'
!uv pip install --python .venv/bin/python "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
# Sanity check — fail fast if Unsloth still isn't importable from the venv.
!uv run python -c "import unsloth; print('[notebook] unsloth importable:', unsloth.__version__)"

## 3. Preflight Then Train

Preflight must pass before the base LLM is loaded. Install Unsloth with ``uv pip`` after ``uv sync`` so ``uv run`` sees it.

In [ ]:
!uv run python -m sft.preflight --config config.yaml
!uv run python -m sft.train --config config.yaml

Expected outputs:
- `<namespace>/firewatch-gnn/gnn/batch_NNN.pt`
- `<namespace>/firewatch-gnn/gnn/normalization.json`
- `<namespace>/firewatch-agent-sft/batch_NNN/`